# 02 — XGBoost baseline

Per-hour XGBoost on engineered features. Adapted variants of this approach actually won the PhysioNet 2019 challenge — for tabular ICU data with heavy missingness, gradient boosting is a very strong baseline that the sequence models have to beat to justify their complexity.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data_loader import load_dataset, patient_train_test_split
from src.features import featurize_many
from src.models.xgb_model import XGBSepsisModel, XGBConfig
from src.evaluate import discrimination_metrics, normalised_utility, sweep_threshold_for_utility, threshold_predictions
from src.visualize import plot_risk_trajectory, plot_calibration

In [ ]:
records = load_dataset('../data', subset=5000, seed=0)
train_pool, test = patient_train_test_split(records, test_frac=0.2, seed=0)
train, valid = patient_train_test_split(train_pool, test_frac=0.125, seed=1)
print(len(train), len(valid), len(test))

In [ ]:
train_frame = featurize_many(train)
valid_frame = featurize_many(valid)
test_frame  = featurize_many(test)
print('train rows:', len(train_frame), 'sepsis prev:', train_frame['SepsisLabel'].mean())

In [ ]:
model = XGBSepsisModel(XGBConfig(n_estimators=600, max_depth=6, learning_rate=0.05))
model.fit(train_frame, valid_frame=valid_frame)

In [ ]:
valid_labels, valid_scores = model.predict_per_patient(valid_frame)
best_thr, best = sweep_threshold_for_utility(valid_labels, valid_scores)
print(f'validation normalised utility = {best["normalised_utility"]:.4f} at threshold = {best_thr:.3f}')

In [ ]:
test_labels, test_scores = model.predict_per_patient(test_frame)
preds = threshold_predictions(test_scores, best_thr)
util = normalised_utility(test_labels, preds)
disc = discrimination_metrics(test_labels, test_scores)
print('Test utility:', util)
print('Test discrimination:', disc)

## Top features
Roughly what we expect: SOFA components, lactate, the time-since-last-measurement channels, and the rolling stats of vitals.

In [ ]:
imp = model.feature_importance().head(25)
fig, ax = plt.subplots(figsize=(7, 6))
imp[::-1].plot.barh(ax=ax, color='#2563eb'); ax.set_xlabel('Gain'); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_calibration(test_labels, test_scores, ax=ax, label='xgb'); plt.show()

In [ ]:
septic = [i for i, l in enumerate(test_labels) if l.any()]
fig, axes = plt.subplots(3, 1, figsize=(9, 8))
for ax, idx in zip(axes, septic[:3]):
    plot_risk_trajectory(test_labels[idx], test_scores[idx], threshold=best_thr, ax=ax)
plt.tight_layout(); plt.show()